In [1]:
# !pip install langchain langchain-community langchain-core
# !pip install chromadb sentence-transformers
# !pip install gitpython
# !pip install ollama

In [2]:
import os
from git import Repo

repo_url = "https://github.com/dmitriypatlasov/ai-driven-financial-analysis.git"
repo_path = "./ai-finance-repo"

if not os.path.exists(repo_path):
    Repo.clone_from(repo_url, repo_path)

print("Repo cloned!")

Repo cloned!


В работе задействованы следующие программные компоненты:

- Модуль os — стандартный модуль языка Python, предназначенный для взаимодействия с операционной системой. В данном случае он используется для проверки существования каталога посредством функции os.path.exists.

- Библиотека GitPython (импортируется класс Repo) — представляет собой объектно-ориентированную обёртку над системой контроля версий Git, предоставляющую разработчику возможность программного выполнения типовых операций, таких как клонирование, фиксация изменений и управление ветками.

Алгоритм работы скрипта сводится к следующему: на первом этапе осуществляется проверка наличия каталога ./ai-finance-repo в локальной файловой системе. В случае его отсутствия выполняется клонирование репозитория ai-driven-financial-analysis посредством вызова метода Repo.clone_from. По завершении процедуры в стандартный поток вывода передаётся диагностическое сообщение, подтверждающее успешное выполнение операции.

In [3]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader

course_path = "./ai-finance-repo/AI Course"

documents = []

# --- TEXT FILES ---
text_loader = DirectoryLoader(
    course_path,
    glob="**/*.(md|py|txt|ipynb)",
    loader_cls=lambda p: TextLoader(p, encoding="utf-8"),
    use_multithreading=True
)
documents.extend(text_loader.load())

# --- PDF FILES ---
pdf_loader = DirectoryLoader(
    course_path,
    glob="**/*.pdf",
    loader_cls=PyPDFLoader,
    use_multithreading=True
)
documents.extend(pdf_loader.load())

print("Total docs:", len(documents))

Total docs: 423


<h>Используемые программные компоненты</h>
<p>Классы <b>DirectoryLoader</b>, <b>TextLoader</b> и <b>PyPDFLoader</b> импортируются из модуля <code>langchain_community.document_loaders</code> и представляют собой загрузчики документов различных форматов. Класс <b>DirectoryLoader</b> обеспечивает обход файловой системы по заданному шаблону, <b>TextLoader</b> предназначен для чтения текстовых файлов, а <b>PyPDFLoader</b> осуществляет извлечение текстового содержимого из PDF-документов.</p>

<h>Алгоритм работы</h>
<p>На первом этапе инициализируется пустой список <code>documents</code>, предназначенный для агрегации загруженных документов. Далее создаётся экземпляр класса <b>DirectoryLoader</b> для обработки текстовых материалов. Параметр <code>glob</code> определяет шаблон поиска файлов с расширениями <code>.md</code>, <code>.py</code>, <code>.txt</code> и <code>.ipynb</code>, что соответствует типичным форматам хранения исходного кода и документации. Параметр <code>use_multithreading=True</code> активирует многопоточный режим загрузки, существенно ускоряющий обработку за счёт параллельного ввода-вывода.</p>

<p>На втором этапе аналогичным образом создаётся загрузчик для PDF-документов, использующий специализированный класс <b>PyPDFLoader</b>. Результаты загрузки обоих типов файлов последовательно добавляются в общий список <code>documents</code> посредством метода <code>extend</code>. По завершении процедуры в стандартный поток вывода передаётся диагностическое сообщение, содержащее общее число загруженных документов.</p>

<p>Таким образом, в работе реализуется унифицированный механизм извлечения гетерогенного контента из файловой системы, что является типичным подготовительным этапом в конвейерах обработки естественного языка и построения систем Retrieval-Augmented Generation.</p>

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

len(chunks)

471

<h>Используемые программные компоненты</h>
<p>Класс <b>RecursiveCharacterTextSplitter</b> импортируется из модуля <code>langchain_text_splitters</code> и представляет собой адаптивный алгоритм рекурсивного разбиения текста на основе символов. В отличие от простых сплиттеров, данный класс последовательно применяет иерархию разделителей — от абзацев к предложениям и далее к словам — что позволяет сохранять семантическую целостность фрагментов и избегать разрывов на смысловых границах.</p>

<h>Алгоритм работы</h>
<p>На первом этапе создаётся экземпляр сплиттера с двумя ключевыми параметрами. Параметр <code>chunk_size=800</code> задаёт максимальный размер итогового фрагмента в символах, а параметр <code>chunk_overlap=100</code> определяет величину перекрытия между соседними фрагментами. Перекрытие необходимо для сохранения контекстных связей: при разрыве связанной информации между фрагментами наличие общей области гарантирует, что семантически зависимые части не окажутся разделёнными.</p>

<p>На втором этапе вызывается метод <code>split_documents</code>, который применяет описанную стратегию разбиения ко всему ранее загруженному массиву документов. Результатом выполнения является список <code>chunks</code>, содержащий фрагменты исходных текстов с сохранёнными метаданными. Финальное выражение <code>len(chunks)</code> возвращает общее количество полученных фрагментов, что позволяет оценить степень детализации корпуса для последующего индексирования.</p>

<p>Таким образом, в работе реализуется этап чанкинга — структурной декомпозиции корпуса документов, обеспечивающий баланс между объёмом контекста, передаваемого в языковую модель, и точностью последующего семантического поиска.</p>


In [5]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-large"
)

/tmp/ipykernel_2384697/2292277092.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

<h>Используемые программные компоненты</h>
<p>Класс <b>HuggingFaceEmbeddings</b> импортируется из модуля <code>langchain_community.embeddings</code> и представляет собой адаптер библиотеки LangChain к экосистеме Hugging Face, позволяющий использовать предобученные модели эмбеддингов из репозитория Hugging Face Hub непосредственно в конвейерах обработки естественного языка.</p>

<h>Алгоритм работы</h>
<p>На первом этапе создаётся экземпляр класса <b>HuggingFaceEmbeddings</b> с единственным обязательным параметром — <code>model_name</code>.</p>

<p>Результатом выполнения фрагмента является объект <code>embedding_model</code>, готовый к использованию в методах <code>embed_documents</code> и <code>embed_query</code> для преобразования текстовых фрагментов и пользовательских запросов в числовые векторы, обеспечивая тем самым математическую основу для последующего поиска по сходству в векторном пространстве.</p>


In [6]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./chroma_db"
)

vectorstore.persist()

/tmp/ipykernel_2384697/2796319596.py:9: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


<h>Используемые программные компоненты</h>
<p>Класс <b>Chroma</b> импортируется из модуля <code>langchain_community.vectorstores</code> и представляет собой программный интерфейс к векторной базе данных Chroma — открытому хранилищу эмбеддингов, оптимизированному для задач семантического поиска и Retrieval-Augmented Generation. Chroma поддерживает автоматическое индексирование векторов, метаданных и исходных документов.</p>

<h>Алгоритм работы</h>
<p>На первом этапе вызывается класс-метод <code>from_documents</code>, выполняющий полный цикл построения индекса. Метод принимает три ключевых параметра: <code>documents</code> — ранее подготовленный список фрагментов, <code>embedding</code> — экземпляр модели эмбеддингов, используемый для преобразования текстовых фрагментов в числовые векторы фиксированной размерности, и <code>persist_directory</code> — путь к каталогу, в котором будет размещено хранилище.</p>

<p>Внутри метода <code>from_documents</code> последовательно выполняются следующие операции: каждый фрагмент текста преобразуется моделью эмбеддингов в векторное представление, после чего вектор вместе с исходным текстом и метаданными записывается в структуру индекса Chroma. По умолчанию используется механизм приближённого поиска ближайших соседей, что обеспечивает суб-линейное время выполнения запросов даже на больших коллекциях документов.</p>

<p>На втором этапе вызывается метод <code>persist</code>, осуществляющий финализацию записи и сохранение всех данных векторного хранилища на диск в каталог <code>./chroma_db</code>. Благодаря этому при последующих запусках приложения отпадает необходимость повторного вычисления эмбеддингов: индекс может быть загружен из файловой системы, что существенно сокращает время инициализации.</p>

<p>Таким образом, в работе реализуется завершающий этап подготовки Retrieval-Augmented Generation системы — создание персистентного семантического индекса учебных материалов, обеспечивающего быстрый и релевантный отклик на пользовательские запросы.</p>


In [7]:
from langchain_community.llms import Ollama

llm = Ollama(
    model="gpt-oss:20b",
    temperature=0.7
)

/tmp/ipykernel_2384697/3353911613.py:3: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(


<h>Используемые программные компоненты</h>
<p>Класс <b>Ollama</b> импортируется из модуля <code>langchain_community.llms</code> и представляет собой адаптер библиотеки LangChain к системе Ollama — платформе, предназначенной для развёртывания и обслуживания больших языковых моделей в режиме вызова через унифицированный интерфейс. Ollama абстрагирует детали загрузки моделей, управления памятью и взаимодействия с GPU, предоставляя разработчику единообразный API.</p>

<h>Алгоритм работы</h>
<p>На первом этапе создаётся экземпляр класса <b>Ollama</b> с двумя конфигурационными параметрами. Параметр <code>model</code> определяет идентификатор загружаемой языковой модели. В данном случае используется значение <code>minimax-m3:cloud</code>, указывающее на облачную версию модели, что позволяет делегировать ресурсоёмкие вычисления внешнему серверу и не нагружать локальную вычислительную машину.</p>

<p>Параметр <code>temperature=0.7</code> задаёт уровень стохастичности генерации. Температура является гиперпараметром распределения вероятностей при выборе следующего токена: значение 0.0 соответствует полностью детерминированной выборке наиболее вероятного продолжения, тогда как значение 1.0 и выше приводит к увеличению разнообразия и креативности генерируемого текста. Значение 0.7 представляет собой компромиссный выбор, обеспечивающий достаточную вариативность ответов при сохранении их семантической связности и фактической корректности.</p>

<p>Результатом выполнения фрагмента является объект <code>llm</code>, предоставляющий стандартный интерфейс LangChain для генерации текста и совместимый с последующими компонентами конвейера — цепочками запросов, агентами и Retrieval-Augmented Generation архитектурами. Таким образом, в работе осуществляется подключение генеративного компонента системы искусственного интеллекта к ранее подготовленной инфраструктуре семантического поиска.</p>


In [8]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate

# retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 6})

# форматируем документы
def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

# prompt для анализа курса
prompt = ChatPromptTemplate.from_template("""
Ты — эксперт по анализу образовательных программ в области AI/ML.

Используй только контекст ниже.

Контекст:
{context}

Вопрос:
{question}

Отвечай подробно, структурированно, на русском языке.
""")

# RAG chain
qa_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
)

# список аналитических вопросов
questions = [
    "Какие основные темы рассматриваются в курсе и как они структурированы?",
    "Какой уровень сложности курса (beginner, intermediate, advanced)? Обоснуй.",
    "Какие алгоритмы машинного обучения и нейросетей используются в курсе?",
    "Чего не хватает в курсе, если сравнивать его с топовыми ML программами (Stanford, DeepLearning.AI)?",
    "Насколько курс практико-ориентирован или теоретический?",
    "Есть ли в курсе темы deep learning, transformers или reinforcement learning?",
    "Можно ли после этого курса работать data scientist или quant researcher?",
    "Какие темы раскрыты хорошо, а какие поверхностно?",
    "Какой был бы идеальный порядок изучения этого курса?",
    "Составь улучшенную версию курса с недостающими темами."
]

# запуск всех вопросов
for i, q in enumerate(questions, 1):
    print(f"\n\n================= ВОПРОС {i} =================\n")
    print("Q:", q)
    print("\nA:\n")
    print(qa_chain.invoke(q))



================= ВОПРОС 1 =================

Q: Какие основные темы рассматриваются в курсе и как они структурированы?

A:

**Курс по искусственному интеллекту и машинному обучению (AI/ML) – «Большие языковые модели, их обучение и применение»**

---

## 1. Общая структура курса

| № | Модуль / блок | Краткое содержание | Длительность* |
|---|---------------|--------------------|--------------|
| **I** | Введение в LLM (Large Language Models) | Что такое большие языковые модели, как они «думают», основные принципы работы. | 4 часа |
| **II** | Проблемы классических RNN | Почему простые рекуррентные сети не подходят для современных задач NLP, ограниченность памяти и обучаемости. | 18 часов |
| **III** | Архитектура трансформеров и интерпретация LLM | Подробный разбор трансформер‑архитектуры (Self‑Attention, позиционные эмбеддинги), как они преодолевают ограничения RNN; вопросы прозрачности больших моделей. | 42 часа |
| **IV** | Обучение модели | Методы обучения LLM: pre‑training, fin

<h>Используемые программные компоненты</h>
<p>Из модуля <code>langchain_core.runnables</code> повторно импортируется класс <b>RunnablePassthrough</b>, обеспечивающий прозрачную передачу пользовательского ввода в цепочке обработки. Из модуля <code>langchain_core.prompts</code> импортируется класс <b>ChatPromptTemplate</b>, предоставляющий механизм декларативного описания шаблонов сообщений для диалоговых языковых моделей с использованием подстановочных переменных в фигурных скобках.</p>

<h>Алгоритм работы</h>

<p><b>Формирование ретривера.</b> На первом этапе векторное хранилище Chroma преобразуется в поисковый модуль посредством вызова метода <code>as_retriever</code>. Параметр <code>search_kwargs</code> со значением ключа <code>k</code>, равным 6, определяет количество наиболее релевантных документов, извлекаемых в ответ на каждый запрос. Увеличение данного параметра по сравнению с предыдущими конфигурациями обеспечивает более полный охват контекста при формировании аналитических ответов.</p>

<p><b>Форматирование документов.</b> Вспомогательная функция <code>format_docs</code> осуществляет преобразование списка объектов-документов, возвращённых ретривером, в единую текстовую строку. Содержимое каждого фрагмента объединяется посредством двойного символа переноса строки, что обеспечивает логическое и визуальное отделение фрагментов друг от друга при последующей передаче в языковую модель.</p>

<p><b>Конструирование промпта.</b> Создаётся экземпляр класса <b>ChatPromptTemplate</b> посредством метода <code>from_template</code>. Шаблон содержит системную инструкцию, определяющую экспертную роль языковой модели в области анализа образовательных программ по направлению искусственного интеллекта и машинного обучения, требование опираться исключительно на предоставленный контекст, а также две подстановочные переменные — <code>{context}</code> и <code>{question}</code>, — замещаемые впоследствии извлечёнными документами и пользовательским запросом соответственно. Завершающая инструкция предписывает генерацию подробных структурированных ответов на русском языке.</p>

<p><b>Сборка RAG-цепочки.</b> Производится композиция компонентов посредством оператора конвейеризации <code>|</code>, реализующего функциональную композицию в стиле Unix. На входе цепочки формируется словарь с двумя ключами: <code>context</code>, значением которого является результат последовательного применения ретривера и функции форматирования документов, и <code>question</code>, значением которого выступает исходный пользовательский запрос, передаваемый без изменений компонентом <b>RunnablePassthrough</b>. Далее словарь поступает в шаблон промпта, после чего сформированное сообщение передаётся в языковую модель, инициализированную ранее через интерфейс Ollama.</p>

<p><b>Пакетная обработка запросов.</b> Определяется список <code>questions</code>, содержащий десять аналитических вопросов, охватывающих тематическую структуру курса, уровень сложности, используемые алгоритмы, практическую ориентированность, тематический охват и рекомендации по улучшению. Посредством цикла <code>for</code> с итератором <code>enumerate</code> осуществляется последовательный вызов метода <code>invoke</code> для каждого вопроса. Перед каждым обращением в стандартный поток вывода выводится разделитель с порядковым номером вопроса, формулировка запроса и заголовок для ответа, что обеспечивает читаемость журнала исполнения и удобство последующего анализа результатов.</p>

<p>Таким образом, в работе реализуется полный цикл интеллектуальной обработки учебных материалов: от семантического поиска и формирования контекста до генерации содержательных аналитических ответов на русском языке с последующим автоматизированным опросом модели по широкому спектру аспектов образовательной программы.</p>